# Tamil AI Video Studio (Kaggle 2×T4 · Open-Source Only)
**Ordered Execution Cells:**
1. **GPU & Disk Verification** (`2× Tesla T4 sm_75`, `/kaggle/temp` vs `/kaggle/working` 20 GB cap)
2. **Install Pinned T4-Safe Dependencies** (FP16 / GGUF / SDPA / xFormers — no `bf16`/`fp8`/`FlashAttention-2`)
3. **Fetch Kaggle Secrets** (`HF_TOKEN`, `RCLONE_GDRIVE_TOKEN`, `STUDIO_UI_PASSWORD`)
4. **Configure `rclone` Google Drive OAuth Remote** (`gdrive:TamilVideoStudio`)
5. **Resumable Model Download to `/kaggle/temp`** (with size manifest & speed benchmark)
6. **Per-Stage Smoke Tests & Wan 2.2 vs LTX-2.3 A/B Benchmark** (prints wall time + peak VRAM)
7. **Launch FastAPI Server + `cloudflared` Tunnel** (prints public HTTPS URL + Studio Password)

In [ ]:
# ==============================================================================
# CELL 1: GPU Check (2x Tesla T4 sm_75) & Disk Space Check (df -h)
# ==============================================================================
import os, subprocess, shutil
from pathlib import Path

print('=== 1. NVIDIA GPU CHECK ===')
subprocess.run(['nvidia-smi'], check=False)

try:
    import torch
    print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  -> cuda:{i}: {p.name} | {p.total_memory / (1024**3):.2f} GB VRAM | sm_{p.major}{p.minor}")
except Exception as e:
    print('PyTorch check:', e)

print('\n=== 2. DISK QUOTA CHECK (/kaggle/temp vs /kaggle/working 20 GB Cap) ===')
Path('/kaggle/temp/tamil_studio/models').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/tamil_studio_out').mkdir(parents=True, exist_ok=True)
subprocess.run(['df', '-h', '/kaggle/temp', '/kaggle/working'], check=False)


In [ ]:
# ==============================================================================
# CELL 2: Install Pinned T4 Dependencies (No bf16 / fp8 / FlashAttention-2)
# ==============================================================================
%%bash
set -e
apt-get update -qq && apt-get install -y -qq ffmpeg rclone fonts-noto-core wget > /dev/null

# Install cloudflared binary for zero-config HTTPS public tunnel
if ! command -v cloudflared &> /dev/null; then
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
fi

# Install CUDA-enabled llama-cpp-python (prebuilt wheel for CUDA 12.1/12.4) + pinned T4 packages
pip install -q --upgrade pip
pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
pip install -q -r requirements.txt

# Install AI4Bharat IndicF5, ACE-Step 1.5, and MMAudio from official open-source repos
pip install -q git+https://github.com/AI4Bharat/IndicF5.git --no-deps
if [ ! -d "/kaggle/temp/tamil_studio/models/MMAudio" ]; then
    git clone --depth 1 https://github.com/hkchengrex/MMAudio.git /kaggle/temp/tamil_studio/models/MMAudio
    pip install -q -e /kaggle/temp/tamil_studio/models/MMAudio --no-deps
fi
echo 'All T4-compatible packages installed successfully.'


In [ ]:
# ==============================================================================
# CELL 3: Fetch Secrets from Kaggle Secrets (Never Hard-Coded)
# ==============================================================================
import os
from pipeline.drive import fetch_kaggle_secret

HF_TOKEN = fetch_kaggle_secret('HF_TOKEN')
RCLONE_GDRIVE_TOKEN = fetch_kaggle_secret('RCLONE_GDRIVE_TOKEN')
STUDIO_UI_PASSWORD = fetch_kaggle_secret('STUDIO_UI_PASSWORD', default='tamil-studio-t4')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
if RCLONE_GDRIVE_TOKEN:
    os.environ['RCLONE_GDRIVE_TOKEN'] = RCLONE_GDRIVE_TOKEN
os.environ['STUDIO_UI_PASSWORD'] = STUDIO_UI_PASSWORD

print(f"HF_TOKEN configured:           {'YES' if HF_TOKEN else 'NO (public repos still work)'}")
print(f"RCLONE_GDRIVE_TOKEN configured: {'YES' if RCLONE_GDRIVE_TOKEN else 'NO (using local mirror fallback)'}")
print(f"STUDIO_UI_PASSWORD configured:  YES ({len(STUDIO_UI_PASSWORD)} chars)")


In [ ]:
# ==============================================================================
# CELL 4: Setup rclone OAuth Remote (gdrive:TamilVideoStudio)
# ==============================================================================
import yaml
from pathlib import Path
from pipeline.drive import DriveSyncManager
from pipeline.t4_patches import ensure_workspace_dirs

cfg = yaml.safe_load(Path('configs/studio_config.yaml').read_text(encoding='utf-8'))
paths = ensure_workspace_dirs(cfg)
drive_mgr = DriveSyncManager(cfg, paths['work_root'] / 'drive_mirror')
print('Drive Status:', drive_mgr.get_status())


In [ ]:
# ==============================================================================
# CELL 5: Resumable Model Download to /kaggle/temp/tamil_studio/models
# ==============================================================================
from scripts.download_models import download_models

# Set dry_run=False on Kaggle to download all GGUF & safetensors checkpoints into /kaggle/temp
download_models(dry_run=False, stage_filter='all')


In [ ]:
# ==============================================================================
# CELL 6: Run Per-Stage Smoke Tests & Wan 2.2 vs LTX-2.3 A/B Comparison
# ==============================================================================
from scripts.smoke_test_all import run_smoke_tests

smoke_report = run_smoke_tests(stage='all')


In [ ]:
# ==============================================================================
# CELL 7: Start FastAPI + Uvicorn Server & Cloudflared Public Tunnel
# ==============================================================================
import re, subprocess, threading, time
import uvicorn
from app.main import app
from app.auth import get_expected_token

PORT = 8000
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info'),
    daemon=True,
).start()
time.sleep(2)

print('Starting cloudflared tunnel...')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(60):
    line = proc.stdout.readline()
    if not line:
        break
    m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print('=' * 80)
print(f'TAMIL AI VIDEO STUDIO LIVE URL: {public_url or "http://localhost:8000"}')
print(f'STUDIO ACCESS TOKEN / PASSWORD: {get_expected_token()}')
print('=' * 80)

# Keep cell alive inside Kaggle session
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    proc.terminate()
